# TD-CCP applied workflow

A fleet manager models replacement choices without estimating a transition density in the structural parameter stage. This notebook checks the panel, fits semi-gradient TD-CCP with Algorithm 2 inference, evaluates held-out choices, bootstraps whole fleet histories, studies a lower replacement cost, and reloads the fitted estimator.

In [ ]:
from importlib.metadata import version
import pickle

import numpy as np
from econirl import TDCCP
from econirl.core.reward_spec import RewardSpec
from econirl.core.types import Panel
from econirl.environments import ArrayMDP
from econirl.simulation.synthetic import simulate_panel

print(f"EconIRL {version('econirl')}")

## Build and inspect the fleet panel

The state is a 12-level condition score. Action 0 keeps the component in service. Action 1 replaces it and resets condition. Two action-contrast reward features identify the condition and replacement coefficients.

In [ ]:
n_states = 12
transitions = np.zeros((2, n_states, n_states), dtype=float)
for state in range(n_states):
    transitions[0, state, state] = 0.65
    transitions[0, state, min(state + 1, n_states - 1)] += 0.35
transitions[1, :, 0] = 1.0
condition = np.linspace(0.0, 1.0, n_states)
features = np.zeros((n_states, 2, 2), dtype=float)
features[:, 0, 0] = -condition
features[:, 1, 1] = -1.0
names = ["condition_cost", "replacement_cost"]
truth = np.array([1.5, 2.2])
env = ArrayMDP(
    transitions,
    features,
    theta=truth,
    discount_factor=0.95,
    parameter_names=names,
    seed=20260811,
)
full_panel = simulate_panel(env, n_individuals=140, n_periods=30, seed=20260811)
train = Panel(full_panel.trajectories[:110])
test = Panel(full_panel.trajectories[110:])
actions = np.asarray(train.get_all_actions())
print(f"Training observations: {train.num_observations}")
print(f"Training individuals: {train.num_individuals}")
print(f"Replacement share: {actions.mean():.3f}")
print(f"Held-out observations: {test.num_observations}")

## Fit and diagnose Algorithm 2

The default semi-gradient stage learns the recursive terms from observed transitions. Two-fold cross-fitting and the locally robust correction account for first-stage estimation in the reported uncertainty.

In [ ]:
reward = RewardSpec(features, names=names)
model = TDCCP(
    n_states=n_states,
    discount=0.95,
    utility=reward,
    se_method="robust",
    seed=90211,
    basis_dim=4,
    ccp_method="logit",
    ccp_poly_degree=2,
)
model.fit(train, transitions=transitions)
print(f"Action-contrast rank: {model.diagnostics_['identification']['contrast_rank']}")
print(f"Transition orientation: {model.diagnostics_['transitions']['orientation']}")
print(f"Converged: {model.converged_}")
print()
print(model.summary())

## Check held-out choices

Held-out negative log likelihood evaluates histories that were not used to estimate the choice probabilities or recursive terms.

In [ ]:
test_states = np.asarray(test.get_all_states(), dtype=int)
test_actions = np.asarray(test.get_all_actions(), dtype=int)
test_probabilities = model.predict_proba(test_states)
chosen = test_probabilities[np.arange(test_states.size), test_actions]
heldout_nll = -np.log(np.clip(chosen, 1e-15, 1.0)).mean()
print(f"Held-out negative log likelihood: {heldout_nll:.4f}")
print(f"Prediction rows sum to one: {np.allclose(test_probabilities.sum(axis=1), 1.0)}")

## Bootstrap whole fleet histories

The bootstrap resamples complete individual trajectories. The supplied transition tensor remains fixed.

In [ ]:
bootstrap_model = TDCCP(
    n_states=n_states,
    discount=0.95,
    utility=reward,
    se_method="bootstrap",
    n_bootstrap=19,
    se_seed=90212,
    seed=90211,
    basis_dim=4,
    ccp_method="logit",
    ccp_poly_degree=2,
    cross_fitting=False,
    robust_se=False,
)
bootstrap_model.fit(train, transitions=transitions)
print(f"Successful bootstrap draws: {bootstrap_model.bootstrap_.n_successful}/19")
print(f"Condition-cost interval: {bootstrap_model.bootstrap_.intervals[0].round(3).tolist()}")

## Lower the replacement cost

The intervention lowers the fitted replacement cost by 0.25. TD-CCP uses its stored transition tensor to solve the changed dynamic program.

In [ ]:
changed_cost = model.params_["replacement_cost"] - 0.25
counterfactual = model.counterfactual(replacement_cost=changed_cost)
baseline_rate = model.policy_[:, 1].mean()
changed_rate = counterfactual.counterfactual_policy[:, 1].mean()
mean_policy_change = np.abs(
    counterfactual.counterfactual_policy - counterfactual.baseline_policy
).mean()
print(f"Mean replacement probability before: {baseline_rate:.3f}")
print(f"Mean replacement probability after: {changed_rate:.3f}")
print(f"Mean absolute policy change: {mean_policy_change:.3f}")

## Reload the fitted estimator

A pickle round trip preserves the summary, encoded-state machinery, predictions, and counterfactual inputs within the same EconIRL minor release.

In [ ]:
restored = pickle.loads(pickle.dumps(model))
prediction_gap = np.max(
    np.abs(restored.predict_proba(np.arange(n_states)) - model.predict_proba(np.arange(n_states)))
)
print(f"Stored EconIRL version: {restored.econirl_version_}")
print(f"Summary preserved: {restored.summary() == model.summary()}")
print(f"Maximum prediction gap: {prediction_gap:.1e}")

## Manager interpretation

The fitted replacement response is useful only when the identification checks pass, held-out choice loss is credible, and the trajectory bootstrap is stable. The counterfactual translates the estimated structural costs into a change in replacement behavior. It remains conditional on the supplied transition tensor and fixed discount factor.